# 卷积神经网络（下）残差连接与 ResNet

本节验证一个经典结论：**网络变深后纯堆叠 Conv 反而难训，残差连接可以化解**。

做法：在 CIFAR-10 小子集上用两个深度相同的网络对比——**Plain-Net**（8 个 Conv 直接堆）vs **ResNet-style**（同样 8 个 Conv，但每 2 个 conv 加一个 skip 连接）。

## 1. CIFAR-10 子集

完整 CIFAR-10 有 50k 训练、10k 测试。为了 notebook 跑得快，只用 5000/1000 子集；演示完结论后，full data + 更多 epoch 才是实际跑法。

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

CACHE = os.path.expanduser('~/.cache/torch_data')
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
train_full = datasets.CIFAR10(CACHE, train=True,  download=True, transform=tfm)
test_full  = datasets.CIFAR10(CACHE, train=False, download=True, transform=tfm)

torch.manual_seed(0)
tr_idx = torch.randperm(len(train_full))[:5000].tolist()
te_idx = torch.randperm(len(test_full))[:1000].tolist()
train_set = Subset(train_full, tr_idx)
test_set  = Subset(test_full,  te_idx)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256)

classes = train_full.classes
print(f'train {len(train_set)}  test {len(test_set)}  classes {classes}')

看几张样本：

In [ ]:
x, y = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for ax, img, lab in zip(axes, x[:8], y[:8]):
    img_disp = img * torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1) + \
               torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    ax.imshow(img_disp.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(classes[lab.item()], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Plain vs Residual block

两种 block 都包含两个 `3×3 Conv + BN + ReLU`；差别只是 ResBlock 加了一条从 block 输入直连到 block 输出的 skip。

In [ ]:
class PlainBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        return x


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        # 当输入/输出通道数 or 尺寸不一致时，shortcut 用 1x1 conv 投影
        self.shortcut = nn.Identity() if (in_ch == out_ch and stride == 1) else \
                        nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                      nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))      # ← 加 skip 连接

## 3. 同样深度的两个网络

深度 = stem (1 Conv) + 3 stage × 2 block × 2 conv = 13 Conv 层。Plain / Res 网络只在 block 类型上不同。

In [ ]:
class Net(nn.Module):
    def __init__(self, block, n_class=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(block(16, 16, stride=1), block(16, 16, stride=1))
        self.stage2 = nn.Sequential(block(16, 32, stride=2), block(32, 32, stride=1))
        self.stage3 = nn.Sequential(block(32, 64, stride=2), block(64, 64, stride=1))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, n_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

for name, block in [('Plain', PlainBlock), ('Res', ResBlock)]:
    n = sum(p.numel() for p in Net(block).parameters())
    print(f'{name}-Net params: {n:,}')

## 4. 训练 + 比较

两个网络用相同的优化器、相同的随机种子、相同的 epoch 数。CPU 上 5 epoch 大约 2-3 分钟一个网络。

In [ ]:
def train_one(model, epochs=5, lr=0.01):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    loss_fn = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'test_acc': []}
    for epoch in range(epochs):
        model.train()
        running, n = 0.0, 0
        for x, y in train_loader:
            opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
            running += loss.item() * x.size(0); n += x.size(0)
        train_loss = running / n
        model.eval(); correct = total = 0
        with torch.no_grad():
            for x, y in test_loader:
                correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
        acc = correct / total
        hist['train_loss'].append(train_loss); hist['test_acc'].append(acc)
        print(f'  epoch {epoch+1}: loss={train_loss:.4f}  test_acc={acc:.4f}')
    return hist

print('=== Plain-Net ===')
torch.manual_seed(0)
plain_hist = train_one(Net(PlainBlock))

print('=== ResNet-style ===')
torch.manual_seed(0)
res_hist = train_one(Net(ResBlock))

### 曲线对比

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(plain_hist['train_loss'], '-o', label='Plain')
ax1.plot(res_hist['train_loss'],   '-o', label='ResNet')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train CE loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(plain_hist['test_acc'], '-o', label='Plain')
ax2.plot(res_hist['test_acc'],   '-o', label='ResNet')
ax2.set_xlabel('epoch'); ax2.set_ylabel('test accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

残差连接的几个直观解释：

- **梯度路径更短**：skip 让早层梯度可以直接从最后一层回传，缓解梯度消失。
- **学习残差函数 $\mathcal{F}(x) = H(x) - x$ 比直接学习 $H(x)$ 容易**：当真函数接近 identity 时，残差只要学接近 0 的小修正即可。
- 在小数据 + 浅网络（这里 13 层）上差距可能不夸张；越深的网络（ResNet-50/100+）差距越明显。

## 5. `torchvision.models.resnet18` API

torchvision 内置了 ResNet18/34/50/101/152 和预训练权重，实际项目里直接用：

In [ ]:
from torchvision.models import resnet18

# 从零开始训练（CIFAR-10 标签 10 类）
m = resnet18(weights=None, num_classes=10)
n_params = sum(p.numel() for p in m.parameters())
print(f'torchvision resnet18 params: {n_params:,}')

# 注意：原版 ResNet18 接受 224x224 输入；CIFAR-10 32x32 会一直被 stride=2 折掉
# 实战中通常会改第一个 Conv (kernel 7→3, stride 2→1) 并去掉首个 MaxPool
out = m(torch.zeros(1, 3, 224, 224))
print(f'output shape on 224x224 input: {tuple(out.shape)}')

## 小结

- **残差连接 (skip connection)** 让深网络更易优化，是 ResNet/DenseNet/Transformer 等架构的基础。
- **BatchNorm** 几乎总是和现代 Conv 网络配合（`Conv → BN → ReLU` 是标配三件套）。
- 写自己的 ResNet：注意 stride>1 或通道数变化时，shortcut 要用 `1×1 Conv` 投影对齐。
- 实战用 `torchvision.models.resnet18(weights=...)` 加预训练 ImageNet 权重 → 微调，比从零训快得多、效果好得多（chap7 会展开学习率 / 优化器策略）。